In [1]:
# Cell 1 - Setup & Install
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance", "plotly", "pandas", "-q"])

import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

print("All packages imported successfully.")

All packages imported successfully.


In [2]:
# Cell 2 - Configuration

TICKERS = [
    "AZZA3.SA", "B3SA3.SA", "BBAS3.SA", "BBDC4.SA", "BBSE3.SA",
    "BPAC11.SA", "CMIN3.SA", "CMIG4.SA", "CPLE3.SA", "CSAN3.SA",
    "DIRR3.SA", "EGIE3.SA", "ABEV3.SA", "EQTL3.SA", "GOAU4.SA",
    "ITSA4.SA", "ITUB4.SA", "KLBN11.SA", "LREN3.SA", "PETR4.SA",
    "PRIO3.SA", "PSSA3.SA", "RENT3.SA", "SAPR11.SA", "SBSP3.SA",
    "SUZB3.SA", "TAEE11.SA", "TOTS3.SA", "VALE3.SA", "VIVT3.SA",
    "WEGE3.SA"
]

START_DATE = "2022-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

SMA_PERIODS = [50, 150, 200]

print(f"Tracking {len(TICKERS)} tickers from {START_DATE} to {END_DATE}")
print(f"SMA periods: {SMA_PERIODS}")

Tracking 31 tickers from 2022-01-01 to 2026-02-13
SMA periods: [50, 150, 200]


In [3]:
# Cell 3 - Data Download & Processing

print(f"Downloading daily data for {len(TICKERS)} tickers...")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, group_by="ticker", auto_adjust=True)
print(f"Download complete. Shape: {raw.shape}")

stock_data = {}
failed = []

for ticker in TICKERS:
    try:
        if len(TICKERS) == 1:
            df = raw[["Close", "Volume"]].copy()
        else:
            df = raw[ticker][["Close", "Volume"]].copy()

        df = df.dropna(subset=["Close"])
        if df.empty:
            raise ValueError("No data")

        # Compute SMAs on daily data
        for period in SMA_PERIODS:
            df[f"Close_SMA{period}"] = df["Close"].rolling(window=period).mean()
            df[f"Vol_SMA{period}"] = df["Volume"].rolling(window=period).mean()

        # Resample to weekly (last observation per week)
        weekly = df.resample("W").last().dropna(subset=["Close"])
        stock_data[ticker] = weekly
    except Exception as e:
        failed.append((ticker, str(e)))

print(f"Successfully processed: {len(stock_data)} tickers")
if failed:
    print(f"Failed: {failed}")

[                       0%                       ]

[***                    6%                       ]  2 of 31 completed

[*****                 10%                       ]  3 of 31 completed

[******                13%                       ]  4 of 31 completed

[***********           23%                       ]  7 of 31 completed

...

[**********************94%********************   ]  29 of 31 completed

[*********************100%***********************]  30 of 31 completed



1 Failed download:
[*********************100%***********************]  30 of 31 completed

['BBSE3.SA']: TypeError("'NoneType' object is not subscriptable")


Download complete. Shape: (1029, 156)
Successfully processed: 30 tickers
Failed: [('BBSE3.SA', 'No data')]


In [4]:
# Cell 4 - Dashboard: Per-ticker charts

SMA_COLORS = {50: "#FF6B6B", 150: "#4ECDC4", 200: "#FFD93D"}

for ticker, df in stock_data.items():
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.65, 0.35],
        subplot_titles=(f"{ticker} - Weekly Close", f"{ticker} - Weekly Volume")
    )

    # Top: Close price
    fig.add_trace(
        go.Scatter(x=df.index, y=df["Close"], name="Close",
                   line=dict(color="#1E90FF", width=1.5)),
        row=1, col=1
    )
    for period in SMA_PERIODS:
        col = f"Close_SMA{period}"
        if col in df.columns:
            fig.add_trace(
                go.Scatter(x=df.index, y=df[col], name=f"SMA {period}",
                           line=dict(color=SMA_COLORS[period], width=1, dash="dot")),
                row=1, col=1
            )

    # Bottom: Volume bars
    fig.add_trace(
        go.Bar(x=df.index, y=df["Volume"], name="Volume",
               marker_color="rgba(100,100,200,0.4)"),
        row=2, col=1
    )
    for period in SMA_PERIODS:
        col = f"Vol_SMA{period}"
        if col in df.columns:
            fig.add_trace(
                go.Scatter(x=df.index, y=df[col], name=f"Vol SMA {period}",
                           line=dict(color=SMA_COLORS[period], width=1, dash="dot")),
                row=2, col=1
            )

    fig.update_layout(
        height=600, width=1100,
        title_text=f"{ticker} — Weekly Price & Volume with SMAs",
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
        template="plotly_white"
    )
    fig.update_yaxes(title_text="Price (BRL)", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)

    fig.show()

In [5]:
# Cell 5 - Summary Table

rows = []
for ticker, df in stock_data.items():
    if df.empty:
        continue
    last = df.iloc[-1]
    row = {
        "Ticker": ticker.replace(".SA", ""),
        "Last Close (BRL)": round(last["Close"], 2),
        "Last Volume": int(last["Volume"]) if pd.notna(last["Volume"]) else None,
    }
    for period in SMA_PERIODS:
        val = last.get(f"Close_SMA{period}")
        row[f"Close SMA{period}"] = round(val, 2) if pd.notna(val) else None
        vval = last.get(f"Vol_SMA{period}")
        row[f"Vol SMA{period}"] = int(vval) if pd.notna(vval) else None
    rows.append(row)

summary = pd.DataFrame(rows)

# Style and display
styled = (
    summary.style
    .format({
        "Last Close (BRL)": "{:.2f}",
        "Last Volume": "{:,.0f}",
        "Close SMA50": "{:.2f}", "Close SMA150": "{:.2f}", "Close SMA200": "{:.2f}",
        "Vol SMA50": "{:,.0f}", "Vol SMA150": "{:,.0f}", "Vol SMA200": "{:,.0f}",
    }, na_rep="—")
    .set_caption(f"B3 Stock Summary — Week ending {END_DATE}")
    .set_table_styles([{"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold")]}])
)
styled

,Ticker,Last Close (BRL),Last Volume,Close SMA50,Vol SMA50,Close SMA150,Vol SMA150,Close SMA200,Vol SMA200
0,AZZA3,25.02,"3,070,000",24.79,"3,406,946",27.26,"2,617,522",29.62,"2,708,130"
1,B3SA3,17.36,"39,912,500",14.84,"45,622,012",13.41,"35,250,616",13.43,"37,496,836"
2,BBAS3,26.03,"111,154,300",22.76,"28,934,200",21.54,"32,352,732",22.09,"32,550,230"
3,BBDC4,21.23,"32,620,300",19.28,"33,927,492",17.50,"30,655,789",16.84,"32,360,773"
4,BPAC11,58.50,"10,400,400",55.64,"9,057,098",48.69,"8,633,867",46.39,"8,435,904"
5,CMIN3,5.56,"9,763,800",5.63,"7,600,088",5.31,"5,978,730",5.24,"6,196,909"
6,CMIG4,11.85,"8,343,500",11.09,"13,034,634",10.68,"10,925,997",10.49,"11,196,337"
7,CPLE3,13.96,"23,311,200",12.79,"18,079,602",11.74,"9,657,781",11.48,"8,797,275"
8,CSAN3,6.11,"34,610,700",5.63,"39,855,702",5.98,"33,504,839",6.40,"29,641,249"
9,DIRR3,15.85,"4,760,500",14.56,"9,030,346",14.14,"5,982,141",13.50,"5,887,016"
